In [1]:
!pip install -U sagemaker==2.*

19.66s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [2]:
import sagemaker
role = sagemaker.get_execution_role()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
print(sagemaker.__version__)

2.255.0


In [4]:
from sagemaker.huggingface import HuggingFace, HuggingFaceModel

In [9]:
huggingface_estimator = HuggingFace(
    entry_point='train.py',
    source_dir='src',
    instance_type='ml.g5.4xlarge',
    instance_count=1,
    role=role,
    transformers_version='4.56.2',
    pytorch_version='2.8.0',
    py_version='py312',
    base_job_name='finetune-llm',
    hyperparameters={
    }
)

huggingface_estimator.fit()

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: finetune-llm-2025-12-19-19-13-59-780


2025-12-19 19:14:00 Starting - Starting the training job
2025-12-19 19:14:00 Pending - Training job waiting for capacity............
2025-12-19 19:15:40 Pending - Preparing the instances for training...
2025-12-19 19:16:14 Downloading - Downloading the training image.....................
2025-12-19 19:20:06 Training - Training image download completed. Training in progress......bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
CUDA compat package should be installed for NVIDIA driver smaller than 575.57.08
Current installed NVIDIA driver version is 570.195.03
Adding CUDA compat to LD_LIBRARY_PATH
/usr/local/cuda/compat:/usr/local/cuda/compat:/usr/local/lib:/opt/amazon/ofi-nccl/lib/x86_64-linux-gnu:/opt/amazon/openmpi/lib:/opt/amazon/efa/lib:/usr/local/cuda/lib64:/usr/local/lib:/usr/local/cuda/lib64:/opt/amazon/ofi-nccl/lib:/opt/amazon/efa/lib:/opt/amazon/openmpi/lib:/usr/local/cuda/lib64
2025-12-19 19:20:49,204 sagemaker-tra

In [10]:
model_artifact = huggingface_estimator.model_data
print("Model artifact location:", model_artifact)

Model artifact location: s3://sagemaker-eu-north-1-008319781174/finetune-llm-2025-12-19-19-13-59-780/output/model.tar.gz


In [13]:

   

huggingface_model = HuggingFaceModel(
    model_data=huggingface_estimator.model_data,  # output of huggingface_estimator.model_data
    role=role,
     transformers_version='4.51.3',
    pytorch_version='2.6.0',
    py_version='py312',
)

predictor = huggingface_model.deploy(
    initial_instance_count=1,
    instance_type='ml.g4dn.2xlarge'
)

INFO:sagemaker:Creating model with name: huggingface-pytorch-inference-2025-12-19-19-34-19-592
INFO:sagemaker:Creating endpoint-config with name huggingface-pytorch-inference-2025-12-19-19-34-20-326
INFO:sagemaker:Creating endpoint with name huggingface-pytorch-inference-2025-12-19-19-34-20-326


------------!

In [20]:
prompt = """
<bos><start_of_turn>user
You are an assistant responsible for classifying mental health status.
I am divorced and happily remarried. Our blended family of children are in their 20s. My youngest, age 20, continues to call me crazy and favors her dad. He was abusive to me, and I left with injuries. He never wanted kids, and I did. He paid child support only through forced wage earners. Now my daughters favor him after all the sacrifices I made for them the past 15 years as a single mom. I don’t deny them a relationship with their dad, but to be called crazy and then watch them hang out with him hurts me to the very core. I’m not sure how to handle this.<end_of_turn>
<start_of_turn>model
"""


response = predictor.predict({
    "inputs": prompt,
    "parameters": {
        "max_new_tokens": 100,
        "do_sample": False,
        # "temperature": 0
    }
})

generated = response[0]['generated_text']
# assistant_start = generated.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
# reply = generated[assistant_start:].strip().split("<|im_end|>")[0].strip()

print("Assistant:", generated)


Assistant: 
<bos><start_of_turn>user
You are an assistant responsible for classifying mental health status.
I am divorced and happily remarried. Our blended family of children are in their 20s. My youngest, age 20, continues to call me crazy and favors her dad. He was abusive to me, and I left with injuries. He never wanted kids, and I did. He paid child support only through forced wage earners. Now my daughters favor him after all the sacrifices I made for them the past 15 years as a single mom. I don’t deny them a relationship with their dad, but to be called crazy and then watch them hang out with him hurts me to the very core. I’m not sure how to handle this.<end_of_turn>
<start_of_turn>model
Based on what you've described, this sounds like 'parenting'.
